# Data Preprocessing: A Practical Guide

Data preprocessing is a crucial step in the machine learning pipeline. It involves cleaning, transforming, and structuring raw data to make it suitable for training a model. Effective preprocessing can significantly improve a model's performance and accuracy.

This notebook provides a practical walkthrough of the key data preprocessing techniques, applying them to the `employee-data.csv` dataset. It serves as the hands-on counterpart to the concepts explained in the `data-preprocessing.md` document.

## 1. Working with Data in Python: Loading and Initial Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from scipy import stats

# Load the dataset
df = pd.read_csv('employee-data.csv')

# --- Basic Information ---
print("--- Displaying the first 5 rows ---")
print(df.head())

print("\n--- Displaying a summary of the DataFrame ---")
df.info()

print("\n--- Displaying descriptive statistics ---")
print(df.describe())

### Basic DataFrame Selections

In [ ]:
print("\n--- Selecting the 'Name' column ---")
print(df['Name'].head())

print("\n--- Selecting the row at index 2 using .loc ---")
print(df.loc[2])

print("\n--- Selecting the value at row 1, column 'City' using .loc ---")
value_loc = df.loc[1, 'City']
print(f"Value at (1, 'City'): {value_loc}")

print("\n--- Selecting the value at row 1, column 2 using .iloc ---")
value_iloc = df.iloc[1, 2]
print(f"Value at (1, 2): {value_iloc}")

## 2. Data Cleaning

### A. Handling Missing Values

In [ ]:
print("--- Missing Values Before Cleaning ---")
print(df.isnull().sum())

# Fill missing Age with the mean
mean_age = df['Age'].mean()
df['Age'].fillna(mean_age, inplace=True)

# Fill missing Salary with the mean
mean_salary = df['Salary'].mean()
df['Salary'].fillna(mean_salary, inplace=True)

# Fill missing Gender with the mode (most frequent value)
mode_gender = df['Gender'].mode()[0]
df['Gender'].fillna(mode_gender, inplace=True)

print("\n--- Missing Values After Imputation ---")
print(df.isnull().sum())

#### A Note on `inplace=True`
You'll notice the use of `inplace=True` in the code above. In Pandas:
- `inplace=False` (the default) means the operation returns a *new* DataFrame, leaving the original unchanged.
- `inplace=True` modifies the original DataFrame directly and returns `None`.

Let's see a quick example:

In [ ]:
# Creating a DataFrame with a missing value
data_example = {'A': [1, 2, np.nan, 4]}
df_example = pd.DataFrame(data_example)
print("--- Original Example DataFrame ---")
print(df_example)

# inplace=False (default)
df_new = df_example.fillna(0)
print("\n--- Original DataFrame after fillna(inplace=False) ---")
print(df_example) # Original is unchanged
print("\n--- New DataFrame created ---")
print(df_new)

# inplace=True
df_example.fillna(0, inplace=True)
print("\n--- Original DataFrame after fillna(inplace=True) ---")
print(df_example) # Original is now modified

### B. Correcting Data Formatting and Consistency

In [ ]:
# Standardize string columns (strip whitespace and convert to title case)
df['City'] = df['City'].str.strip().str.title()
df['Department'] = df['Department'].str.strip().str.title()

print("--- Unique Cities After Standardization ---")
print(df['City'].unique())

print("\n--- Unique Departments After Standardization ---")
print(df['Department'].unique())

### C. Identifying and Removing Duplicate Records

In [ ]:
print(f"Original number of rows: {len(df)}")
df.drop_duplicates(inplace=True)
print(f"Number of rows after dropping duplicates: {len(df)}")

### D. Outlier Detection

First, let's visualize the distribution of 'Salary' to spot potential outliers.

In [ ]:
# Box plot to visualize outliers in Salary
plt.figure(figsize=(10, 6))
sns.boxplot(x=df['Salary'])
plt.title('Box Plot of Salary to Detect Outliers')
plt.xlabel('Salary')
plt.show()

The box plot above clearly shows one data point far to the right, which is a likely outlier. Now, we can use the Z-score method to programmatically identify and remove it.

In [ ]:
df['Salary_ZScore'] = np.abs(stats.zscore(df['Salary']))
outliers = df[df['Salary_ZScore'] > 3]

print("--- Potential Outliers in Salary (Z-Score > 3) ---")
print(outliers[['Name', 'Salary']])

# Remove the outlier for further analysis
df = df[df['Salary_ZScore'] <= 3].copy()
df.drop('Salary_ZScore', axis=1, inplace=True)

### E. Removing Irrelevant Features

In [ ]:
# The EmployeeID is just an identifier and adds no value to most machine learning models.
df_cleaned = df.drop('EmployeeID', axis=1)
print("--- DataFrame after dropping 'EmployeeID' ---")
print(df_cleaned.head())

## 3. Data Transformation and Scaling

### A. Standardization (Z-Score Normalization)

In [ ]:
scaler_standard = StandardScaler()
df_cleaned['Salary_Standardized'] = scaler_standard.fit_transform(df_cleaned[['Salary']])
print("--- Salary after Standardization ---")
print(df_cleaned[['Salary', 'Salary_Standardized']].head())

### B. Min-Max Scaling (Normalization)

In [ ]:
scaler_minmax = MinMaxScaler()
df_cleaned['YearsExperience_Normalized'] = scaler_minmax.fit_transform(df_cleaned[['YearsExperience']])
print("--- YearsExperience after Min-Max Scaling ---")
print(df_cleaned[['YearsExperience', 'YearsExperience_Normalized']].head())

### C. Mean Normalization

In [ ]:
# Formula: (X - X_mean) / (X_max - X_min)
mean_perf = df_cleaned['PerformanceScore'].mean()
min_perf = df_cleaned['PerformanceScore'].min()
max_perf = df_cleaned['PerformanceScore'].max()

df_cleaned['PerformanceScore_MeanNormalized'] = (df_cleaned['PerformanceScore'] - mean_perf) / (max_perf - min_perf)

print("--- PerformanceScore after Mean Normalization ---")
print(df_cleaned[['PerformanceScore', 'PerformanceScore_MeanNormalized']].head())

## 4. Handling Categorical Data

### A. Label Encoding

This method assigns a unique integer to each category (e.g., 'Single' -> 0, 'Married' -> 1, 'Divorced' -> 2). It's simple and works well for features with a natural order (ordinal data). However, for nominal data (no intrinsic order), it can mislead the model into thinking there's a relationship between the numbers (e.g., that 2 > 1 > 0), which might not be true.

In [ ]:
le = LabelEncoder()
df_cleaned['MaritalStatus_Encoded'] = le.fit_transform(df_cleaned['MaritalStatus'])
print("--- Marital Status after Label Encoding ---")
print(df_cleaned[['MaritalStatus', 'MaritalStatus_Encoded']].head())

### B. One-Hot Encoding

In [ ]:
department_dummies = pd.get_dummies(df_cleaned['Department'], prefix='Dept', dtype=int)
df_final = pd.concat([df_cleaned, department_dummies], axis=1)
df_final.drop('Department', axis=1, inplace=True)

print("--- DataFrame after One-Hot Encoding 'Department' ---")
print(df_final.head())

## 5. Data Visualization

In [ ]:
sns.set_style('whitegrid')

# Histogram of Salary
plt.figure(figsize=(10, 6))
sns.histplot(df_final['Salary'], bins=20, kde=True)
plt.title('Salary Distribution (After Cleaning)')
plt.xlabel('Salary')
plt.ylabel('Frequency')
plt.show()

# Bar chart of employees per department (using the encoded columns)
plt.figure(figsize=(10, 6))
dept_cols = [col for col in df_final.columns if 'Dept_' in col]
dept_counts = df_final[dept_cols].sum()
sns.barplot(x=dept_counts.index, y=dept_counts.values)
plt.title('Number of Employees per Department')
plt.xlabel('Department')
plt.ylabel('Number of Employees')
plt.xticks(rotation=45)
plt.show()

## 6. Conclusion

The data has been cleaned, transformed, and is now ready for machine learning model training. The final, processed data is saved to a new CSV file.

In [ ]:
# Save the cleaned dataframe to a new CSV file
df_final.to_csv('employee-data-cleaned.csv', index=False)

print("--- Final Cleaned DataFrame Head ---")
print(df_final.head())
print("\nCleaned data saved to 'employee-data-cleaned.csv'")